In [2]:
# ==== 3x3 synced video + plots → MP4 ====
# Requirements: `combined` DataFrame already built (one row per OBS frame, with server_time and all synced columns)

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter

# ------------------ CONFIG ------------------
OBS_VIDEO_PATH   = r"/standard/UVA-DSA/MIDAS/Organized/final_data/bt1/video/2024-07-19 12-11-23.mp4"   # <-- set your video file path
OUTPUT_MP4_PATH  = r"/standard/UVA-DSA/MIDAS/Organized/final_data/bt1/synched_data/synched_preview.mp4"  # <-- set your output file path

SYNCHED_CSV_PATH = r"/standard/UVA-DSA/MIDAS/Organized/final_data/bt1/synched_data/synched_all_bt1.csv"  # <-- set your combined CSV path

combined = pd.read_csv(SYNCHED_CSV_PATH)

FPS_OVERRIDE     = None        # None to use video fps; or set e.g. 30
WINDOW_SEC       = 5.0         # trailing window to display (seconds)
USE_DATETIME_X   = False       # True = wall-clock timestamps, False = seconds from start (faster)
MAX_FRAMES       = None        # None = use all frames in `combined`; or set an int to limit

# Choose variables to show together on each subplot:
RAVEN_VARS   = [  # e.g. positions/orientations you care about
    "raven_field.pos0", "raven_field.pos1", "raven_field.pos2"
]
CONSOLE_VARS = [
    "console_pos0", "console_pos1", "console_pos2"
]
TRAKSTAR_VARS = [
    "trakstar_sensor_0_x", "trakstar_sensor_1_x", "trakstar_sensor_2_x", "trakstar_sensor_3_x"
]
SW_AXIS = "x"  # 'x' | 'y' | 'z' -> plots sw_left_<axis> and sw_right_<axis> together
PDS_VARS = [   # mixture of pressures and/or pressed flags
    "Pedal 5 Pressesd"
]
CONSOLE_PEDAL_COL = "console_pedal"

# Figure / video size (pixels). 1920×1080 works well; adjust DPI or size to taste.
FIG_W, FIG_H, DPI = 1920, 1080, 100
# -------------------------------------------

df = combined.copy()
if "server_time" not in df.columns:
    raise ValueError("combined must include 'server_time' (epoch ns).")






# Time axis
df["server_time"] = pd.to_numeric(df["server_time"], errors="coerce").astype("Int64").fillna(0).astype("int64")
t0_ns = int(df["server_time"].iloc[0])
if USE_DATETIME_X:
    X_all = pd.to_datetime(df["server_time"], unit="ns", errors="coerce")
    x_fmt = DateFormatter("%H:%M:%S.%f")
else:
    X_all = (df["server_time"] - t0_ns) / 1e9  # seconds from start
    x_fmt = None

# Helper: mask sentinel -1 to NaN for modalities where -1 means "no data"
def _mask_neg1(series):
    s = pd.to_numeric(series, errors="coerce")
    return s.mask(np.isclose(s, -1))

# Build column groups (filter to existing only)
def _existing(cols):
    return [c for c in cols if c in df.columns]

RAVEN_VARS   = _existing(RAVEN_VARS)
CONSOLE_VARS = _existing(CONSOLE_VARS)
TRAKSTAR_VARS = _existing(TRAKSTAR_VARS)
PDS_VARS     = _existing(PDS_VARS)

sw_left_col  = f"sw_left_{SW_AXIS}"
sw_right_col = f"sw_right_{SW_AXIS}"
SW_VARS      = _existing([sw_left_col, sw_right_col])

# Open video
cap = cv2.VideoCapture(OBS_VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {OBS_VIDEO_PATH}")

video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
fps = FPS_OVERRIDE or video_fps

# Prepare writer (match figure pixel size)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_MP4_PATH, fourcc, fps, (FIG_W, FIG_H))
if not writer.isOpened():
    cap.release()
    raise RuntimeError(f"Cannot open VideoWriter for: {OUTPUT_MP4_PATH}")

# Prepare figure/axes once; we'll clear & redraw each frame
plt.close("all")
fig, axes = plt.subplots(3, 3, figsize=(FIG_W / DPI, FIG_H / DPI), dpi=DPI)
fig.subplots_adjust(wspace=0.25, hspace=0.35)
plt.tight_layout()              # <-- once, here


# Aliases for axes
ax_v1, ax_v2, ax_v3 = axes[0,0], axes[0,1], axes[0,2]
ax_raven, ax_console, ax_trakstar = axes[1,0], axes[1,1], axes[1,2]
ax_sw, ax_pds, ax_cpedal = axes[2,0], axes[2,1], axes[2,2]



from matplotlib import ticker

N_TICKS = 6  # number of x ticks per subplot (keep it constant)

def set_xlim_and_fixed_ticks(ax, x_now, window_sec, use_datetime, xfmt):
    # Set the x-limits to the trailing window
    if use_datetime:
        x_end = pd.to_datetime(x_now)
        x_start = x_end - pd.Timedelta(seconds=window_sec)
        ax.set_xlim(x_start, x_end)

        # Build fixed tick positions anchored to window edges
        xs = pd.date_range(x_start, x_end, periods=N_TICKS)
        ax.xaxis.set_major_locator(ticker.FixedLocator(xs.view('i8')))  # use int64 ns positions
        if xfmt is not None:
            ax.xaxis.set_major_formatter(xfmt)
    else:
        x_end = float(x_now)
        x_start = x_end - window_sec
        ax.set_xlim(x_start, x_end)

        xs = np.linspace(x_start, x_end, N_TICKS)
        ax.xaxis.set_major_locator(ticker.FixedLocator(xs))
        # Optional: pretty labels like -5..0 relative to “now”
        labels = [f"{t - x_end:.0f}" for t in xs]
        ax.xaxis.set_major_formatter(ticker.FixedFormatter(labels))
        ax.set_xlabel("Time (s, relative to frame)")  # or leave blank if you prefer

    # Keep minor ticks off to reduce clutter (optional)
    ax.minorticks_off()



# Style helpers
def style_axis(ax, title):
    ax.set_title(title, fontsize=12)
    ax.grid(True, alpha=0.3)

def set_xlim_to_window(ax, x_end):
    if USE_DATETIME_X:
        #  convert seconds window to timedeltas from x_end
        from pandas import Timedelta
        ax.set_xlim(x_end - pd.Timedelta(seconds=WINDOW_SEC), x_end)
    else:
        ax.set_xlim(x_end - WINDOW_SEC, x_end)

def draw_cursor(ax, x_now):
    if USE_DATETIME_X:
        ax.axvline(x_now, color="k", linewidth=1.0, alpha=0.7)
    else:
        ax.axvline(float(x_now), color="k", linewidth=1.0, alpha=0.7)

# Precompute “time” as numeric seconds for easy masking
T_sec = (df["server_time"] - t0_ns) / 1e9

# Frame loop
n_rows = len(df)
max_frames = min(int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or n_rows), n_rows)
if MAX_FRAMES is not None:
    max_frames = min(max_frames, MAX_FRAMES)

for i in range(max_frames):
    ok, frame_bgr = cap.read()
    if not ok:
        print(f"[WARN] Ran out of video frames at i={i}.")
        break

    # Current time
    t_now_ns = int(df["server_time"].iloc[i])
    if USE_DATETIME_X:
        x_now = pd.to_datetime(t_now_ns, unit="ns")
    else:
        x_now = float(T_sec.iloc[i])

    # Compute mask for window [t_now - WINDOW_SEC, t_now]
    if USE_DATETIME_X:
        # Use T_sec for masking, then map to X_all
        mask = (T_sec >= T_sec.iloc[i] - WINDOW_SEC) & (T_sec <= T_sec.iloc[i])
    else:
        mask = (T_sec >= T_sec.iloc[i] - WINDOW_SEC) & (T_sec <= T_sec.iloc[i])

    # --- Clear axes
    for ax in axes.ravel():
        ax.cla()

    # ------------- Row 1: three copies of video frame -------------
    # Convert to RGB for imshow
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    for ax, title in zip([ax_v1, ax_v2, ax_v3],
                         ["OBS video (1)", "OBS video (2) — replace later", "OBS video (3) — replace later"]):
        ax.imshow(frame_rgb)
        ax.set_axis_off()
        ax.set_title(title, fontsize=12)

    # ------------- Row 2, Col 1: Raven -------------
    style_axis(ax_raven, "Raven kinematics")
    if RAVEN_VARS:
        for c in RAVEN_VARS:
            y = pd.to_numeric(df[c], errors="coerce")
            ax_raven.plot(X_all[mask], y[mask], label=c, linewidth=1.0)
        ax_raven.legend(fontsize=8, loc="upper right")
    if x_fmt: ax_raven.xaxis.set_major_formatter(x_fmt)
    set_xlim_and_fixed_ticks(ax_raven, x_now, WINDOW_SEC, USE_DATETIME_X, x_fmt)
    draw_cursor(ax_raven, x_now)


    # ------------- Row 2, Col 2: Console -------------
    style_axis(ax_console, "Console data")
    if CONSOLE_VARS:
        for c in CONSOLE_VARS:
            y = pd.to_numeric(df[c], errors="coerce")
            ax_console.plot(X_all[mask], y[mask], label=c, linewidth=1.0)
        ax_console.legend(fontsize=8, loc="upper right")
    if x_fmt: ax_console.xaxis.set_major_formatter(x_fmt)
    set_xlim_and_fixed_ticks(ax_console, x_now, WINDOW_SEC, USE_DATETIME_X, x_fmt)
    draw_cursor(ax_console, x_now)

    # ------------- Row 2, Col 3: TrakStar -------------
    style_axis(ax_trakstar, "TrakStar")
    if TRAKSTAR_VARS:
        for c in TRAKSTAR_VARS:
            y = _mask_neg1(df[c]) if c in df.columns else None
            if y is not None:
                ax_trakstar.plot(X_all[mask], y[mask], label=c, linewidth=1.0)
        ax_trakstar.legend(fontsize=8, loc="upper right")
    if x_fmt: ax_trakstar.xaxis.set_major_formatter(x_fmt)
    set_xlim_and_fixed_ticks(ax_trakstar, x_now, WINDOW_SEC, USE_DATETIME_X, x_fmt)
    draw_cursor(ax_trakstar, x_now)

    # ------------- Row 3, Col 1: Smartwatch L/R (same axis) -------------
    style_axis(ax_sw, f"Smartwatch L/R ({SW_AXIS.upper()})")
    if SW_VARS:
        for c in SW_VARS:
            y = _mask_neg1(df[c])
            ax_sw.plot(X_all[mask], y[mask], label=c, linewidth=1.0)
        ax_sw.legend(fontsize=8, loc="upper right")
    if x_fmt: ax_sw.xaxis.set_major_formatter(x_fmt)
    set_xlim_and_fixed_ticks(ax_sw, x_now, WINDOW_SEC, USE_DATETIME_X, x_fmt)
    draw_cursor(ax_sw, x_now)

    # ------------- Row 3, Col 2: PDS -------------
    style_axis(ax_pds, "PDS")
    if PDS_VARS:
        for c in PDS_VARS:
            # pressures: mask -1; pressed flags: step 0/1
            if "Pressed" in c or "Pressesd" in c:
                y = pd.to_numeric(df[c], errors="coerce").fillna(0)
                ax_pds.step(X_all[mask], y[mask], where="post", label=c)
                ax_pds.set_ylim(-0.1, 1.1)
            else:
                y = _mask_neg1(df[c])
                ax_pds.plot(X_all[mask], y[mask], label=c, linewidth=1.0)
        ax_pds.legend(fontsize=8, loc="upper right")
    if x_fmt: ax_pds.xaxis.set_major_formatter(x_fmt)
    set_xlim_and_fixed_ticks(ax_pds, x_now, WINDOW_SEC, USE_DATETIME_X, x_fmt)
    draw_cursor(ax_pds, x_now)

    # ------------- Row 3, Col 3: Console pedal only -------------
    style_axis(ax_cpedal, "Console pedal")
    if CONSOLE_PEDAL_COL in df.columns:
        y = pd.to_numeric(df[CONSOLE_PEDAL_COL], errors="coerce")
        ax_cpedal.plot(X_all[mask], y[mask], linewidth=1.0)
    if x_fmt: ax_cpedal.xaxis.set_major_formatter(x_fmt)
    set_xlim_and_fixed_ticks(ax_cpedal, x_now, WINDOW_SEC, USE_DATETIME_X, x_fmt)
    draw_cursor(ax_cpedal, x_now)

    # Tighten layout
    plt.tight_layout()

    # --- Render fig to image & write to video ---
    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8).reshape(h, w, 3)
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    # Resize to exact writer dims (guard against rounding issues)
    if (w, h) != (FIG_W, FIG_H):
        img_bgr = cv2.resize(img_bgr, (FIG_W, FIG_H), interpolation=cv2.INTER_AREA)

    writer.write(img_bgr)

# Cleanup
cap.release()
writer.release()
plt.close(fig)
print(f"[OK] Wrote video: {OUTPUT_MP4_PATH}")


/tmp/ipykernel_137914/3953033967.py:281: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8).reshape(h, w, 3)
/tmp/ipykernel_137914/3953033967.py:281: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8).reshape(h, w, 3)
/tmp/ipykernel_137914/3953033967.py:281: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8).reshape(h, w, 3)
/tmp/ipykernel_137914/3953033967.py:281: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  img = np.frombuf

[OK] Wrote video: /standard/UVA-DSA/MIDAS/Organized/final_data/bt1/synched_data/synched_preview.mp4


/tmp/ipykernel_137914/3953033967.py:281: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8).reshape(h, w, 3)
